# 04 — SAC Teacher → SFT Dataset (Local)

**Phase 2→3 transition** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

Goal: produce a behavior-cloning dataset for fine-tuning a single 3-building SLM.

1. Train a SAC agent for `SAC_EPISODES` (default 20) on the full 6-building
   district (`BUILDINGS=[0..5]`). Per-building policies — `central_agent=False`.
2. Evaluate it (Phase I + Combined challenge scores).
3. Run the trained SAC deterministically for **one full year (8 760 steps)**
   on the same 6-building env, and dump `(state_text, action_tokens)` pairs as
   JSONL — but slice each step into TWO rows: one for `TRAINING_BUILDINGS=[0,1,2]`
   and one for `HELDOUT_BUILDINGS=[3,4,5]`. The student SLM learns a 3-building
   policy; at Phase 4 the same LoRA loads into both agents.
4. Ship the JSONL to Colab → notebook 05 does LoRA SFT on Gemma.

Runs on the MacBook (CPU). Notebook 05 (Colab) does the GPU work.


## 0 — Config

In [1]:
import sys, time, warnings, pickle, copy
from pathlib import Path

import numpy as np
import pandas as pd

# Make src/ importable
_ROOT = Path.cwd().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

warnings.filterwarnings("ignore")

# ── Hyperparameters ────────────────────────────────────────────────────────
SAC_EPISODES   = 20            # teacher training budget
REWARD_FN      = "merlin"     # 'merlin' or 'eco' — must match notebook 05 eval
ARTIFACTS      = _ROOT / "notebooks" / "artifacts"
AGENT_DIR      = ARTIFACTS / "agents"
DATASET_DIR    = ARTIFACTS / "sft_datasets"
AGENT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

STAMP = time.strftime("%Y%m%d_%H%M%S")
print(f"Run stamp: {STAMP}")

Run stamp: 20260509_235739


## 1 — Imports

In [2]:
import citylearn
from citylearn.agents.sac import SAC
from citylearn.agents.rbc import BasicBatteryRBC

from src.env  import (
    make_env, snapshot_state, SEED,
    BUILDINGS,           # full 6-building district (SAC teacher trains here)
    TRAINING_BUILDINGS,  # [0,1,2] — Phase 3 single-agent slice
    HELDOUT_BUILDINGS,   # [3,4,5] — second slice for 2x data + generalization test
)
from src.eval import evaluate, comparison_table
from src.sft  import (
    render_state, action_to_token, format_action_block,
    dump_sac_trajectory_jsonl, make_sft_prompt,
)

np.random.seed(SEED)
print(f"CityLearn {citylearn.__version__}")
print(f"SAC teacher trains on BUILDINGS={BUILDINGS} (6-bldg district)")
print(f"SFT dataset emits 3-bldg slices: {TRAINING_BUILDINGS} and {HELDOUT_BUILDINGS}")


CityLearn 2.6.0b2  |  buildings=[0, 1, 2, 3, 4, 5]


## 2 — Train SAC (20 episodes)

20 episodes is a smoke-test budget — we expect the teacher to be only modestly
above RBC. The pipeline still works end-to-end; for a stronger teacher, raise
`SAC_EPISODES` and rerun.

In [3]:
env_sac   = make_env(reward_fn=REWARD_FN, session_name=f"sac_{STAMP}")
agent_sac = SAC(env=env_sac, seed=SEED)

print(f"Training SAC — {SAC_EPISODES} episodes (last deterministic) …")
_t0 = time.time()
agent_sac.learn(episodes=SAC_EPISODES, deterministic_finish=True)
print(f"SAC training: {time.time() - _t0:.1f}s")

Training SAC — 20 episodes (last deterministic) …
SAC training: 8740.5s


## 3 — Evaluate SAC vs RBC

In [4]:
# RBC reference (1 episode)
env_rbc   = make_env(reward_fn=REWARD_FN, session_name=f"rbc_{STAMP}")
agent_rbc = BasicBatteryRBC(env=env_rbc)
agent_rbc.learn(episodes=1)

res_rbc = evaluate(env_rbc, label="RBC")
res_sac = evaluate(env_sac, label=f"SAC ({SAC_EPISODES}ep)")

challenge_df, zne_df = comparison_table([res_rbc, res_sac])
print("Challenge scores (lower is better):")
display(challenge_df)
print("\nZNE:")
display(zne_df[["ZNE ratio (solar / import)", "self-consumption ratio"]])

Challenge scores (lower is better):


,C — cost,G — carbon,R — ramping,1L — load factor,Phase I (C+G)/2,Combined (C+G+D)/3
RBC,0.9173,0.9671,1.0743,0.9462,0.9422,0.9649
SAC (20ep),0.8425,0.8163,0.7497,0.9528,0.8294,0.8367



ZNE:


,ZNE ratio (solar / import),self-consumption ratio
RBC,1.1757,0.6678
SAC (20ep),1.3351,0.8039


## 4 — Save SAC Agent (inference-only pickle)

In [5]:
agent_inf = copy.deepcopy(agent_sac)
for buf in agent_inf.replay_buffer:
    buf.buffer.clear()

sac_path = AGENT_DIR / f"sac_{REWARD_FN}_inference_{STAMP}.pkl"
with open(sac_path, "wb") as f:
    pickle.dump(agent_inf, f)
print(f"SAC inference agent saved: {sac_path}  ({sac_path.stat().st_size / 1e6:.1f} MB)")

SAC inference agent saved: /Users/antonisbast/projects/eclipse-thesis/notebooks/artifacts/agents/sac_merlin_inference_20260509_235739.pkl  (105.5 MB)


## 5 — Dump 1-Year Trajectory as SFT JSONL (3-bldg slices)

We roll the trained SAC out for one full episode (8 760 hourly steps) on the
full 6-building district. At every step we emit **two** JSONL rows — one for
slice `[0,1,2]` and one for `[3,4,5]` — by selecting the relevant buildings
from the snapshot and the corresponding SAC actions.

**Why slice instead of training SAC on 3 buildings?**
SAC is per-building (`central_agent=False`), so each building's policy is
independent — slicing introduces zero distribution mismatch and saves a Colab
session. **Why two slices?** The SLM learns a building-agnostic 3-building
policy: at Phase 4 deployment the same LoRA loads into agent α (sees B0–2)
and agent β (sees B3–5) without retraining.

Each row:
- **prompt**: `STATE:\n` + `render_state(snapshot[slice])` — exactly what the
  SLM sees at inference time on its 3-building slice.
- **response**: 3 `<action building=i>TOKEN</action>` lines (i ∈ {0,1,2} —
  always local indices in the slice, not global).
- **slice**: the global building indices this row was emitted for.


In [6]:
# Fresh env for the dump (resets to t=0) — SAC needs the full 6-building env
# it was trained on; slicing happens at JSONL emission time.
env_dump = make_env(
    buildings=BUILDINGS,
    reward_fn=REWARD_FN,
    session_name=f"sac_dump_{STAMP}",
)

jsonl_path = DATASET_DIR / f"sac_{REWARD_FN}_distill_{STAMP}.jsonl"

_t0 = time.time()
stats = dump_sac_trajectory_jsonl(
    env=env_dump,
    agent=agent_sac,
    out_path=jsonl_path,
    snapshot_fn=snapshot_state,
    building_slices=[TRAINING_BUILDINGS, HELDOUT_BUILDINGS],
)
print(f"Dumped {stats['n_steps']} env steps × {stats['n_slices']} slices "
      f"= {stats['n_rows']} rows to {stats['path']}")
print(f"  per-row buildings = {stats['n_buildings']}  |  elapsed {time.time() - _t0:.1f}s")


Dumped 8759 steps to /Users/antonisbast/projects/eclipse-thesis/notebooks/artifacts/sft_datasets/sac_merlin_distill_20260509_235739.jsonl  in 26.0s


## 6 — Inspect Dataset (sanity check)

In [7]:
import json

lines = jsonl_path.read_text().strip().split("\n")
print(f"Total examples: {len(lines)}  (expect ~2× n_steps when using 2 slices)")

# Show one row from each slice near mid-year
mid = len(lines) // 2
for offset in (0, 1):
    row = json.loads(lines[mid + offset])
    print(f"\n── Sample row {mid+offset} (slice={row.get('slice')}, t={row.get('t')}) ──")
    print("PROMPT:")
    print(row["prompt"])
    print("\nRESPONSE:")
    print(row["response"])

# Action token distribution — sanity on bucket spread
from collections import Counter
import re
tok_re = re.compile(r"<action building=\d+>([A-Z_0-9]+)</action>")
all_toks = []
for ln in lines:
    all_toks.extend(tok_re.findall(json.loads(ln)["response"]))
print(f"\nAction-token distribution (top 12 of {len(all_toks)} tokens):")
for tok, n in Counter(all_toks).most_common(12):
    print(f"  {tok:14s} {n:6d}")


Total examples: 8759

── Sample at t≈4380 ──
PROMPT:
STATE:
Month 1, Mon 11:00  |  price=0.210 (LOW)  |  carbon=0.171 (MID)
Forecast:  price+6h=PEAK  price+12h=LOW  solar+6h=LOW
Buildings:
  B0: SoC= 53.4%  load=0.32 kWh  last_net=-0.61 kWh  solar=HIGH
  B1: SoC= 36.9%  load=0.32 kWh  last_net=+0.01 kWh  solar=HIGH
  B2: SoC= 42.0%  load=0.16 kWh  last_net=-0.16 kWh  solar=HIGH
  B3: SoC= 14.5%  load=2.27 kWh  last_net=+0.45 kWh  solar=HIGH
  B4: SoC= 50.0%  load=0.95 kWh  last_net=+0.32 kWh  solar=HIGH
  B5: SoC= 99.1%  load=0.00 kWh  last_net=-2.41 kWh  solar=HIGH

RESPONSE:
<action building=0>CHARGE_40</action>
<action building=1>CHARGE_40</action>
<action building=2>CHARGE_40</action>
<action building=3>CHARGE_20</action>
<action building=4>CHARGE_40</action>
<action building=5>CHARGE_20</action>

Action-token distribution (top 12 of 52554 tokens):
  DISCHARGE_20     21424  (40.8%)
  DISCHARGE_40     13511  (25.7%)
  CHARGE_20         6543  (12.5%)
  IDLE              6439  (12.3%)

## 7 — Show the SFT prompt template (used in notebook 05)

Notebook 05 will wrap each row as:

```
<system> make_sft_prompt(3) </system>
<user>   STATE:\n... </user>            ← row["prompt"]
<assistant> <action building=0>...   </assistant>  ← row["response"]
```

Each row covers a 3-building slice — local building indices 0–2 inside the
prompt/response always refer to positions within the slice, not global B0–B5.


In [8]:
print(make_sft_prompt(3))

You are an energy management agent for 6 buildings. Goal: minimize grid dependency and energy costs over time.

[Actions] — choose exactly one per building:
CHARGE_100, CHARGE_80, CHARGE_60, CHARGE_40, CHARGE_20, IDLE, DISCHARGE_20, DISCHARGE_40, DISCHARGE_60, DISCHARGE_80, DISCHARGE_100

[State Variables & Environment]
- 'price': Current cost of grid electricity. PEAK indicates high cost.
- 'solar': Renewable energy generated locally.
- 'load': Energy demanded by the building's operations.
- 'SoC': Battery State of Charge (0% = empty, 100% = full).
- Charging stores energy (efficient when solar HIGH or price LOW).
- Discharging serves load and reduces grid import (best when price PEAK and SoC sufficient).
- Forecast fields show conditions 6 or 12 hours ahead.

[Output Format]
Output exactly 6 action lines, one per building, and nothing else:
<action building=0>YOUR_CHOICE</action>
<action building=1>YOUR_CHOICE</action>
<action building=2>YOUR_CHOICE</action>
<action building=3>YOUR_C

## 8 — Hand-off to Colab

**Commit and push** the JSONL artifact (it's a few MB) so notebook 05
can `git pull` it in Colab. Or upload manually via the Files panel.

```bash
git add notebooks/artifacts/sft_datasets/sac_*.jsonl
git commit -m "sac distill dataset (20ep teacher, full year)"
git push
```

Then open `notebooks/05_sft_gemma_colab.ipynb` in Colab.